In [16]:
import langchain

In [29]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-MiniLM-L6-v2"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vishesh1412/celebrity-face-image-dataset")

print("Path to dataset files:", path)

In [44]:
from torchvision import datasets
from PIL import Image

# Your dataset path
dataset_path = r"D:\Facial-Media-Recognition-System\Dataset"

# Load using ImageFolder (gets image paths + labels)
dataset = datasets.ImageFolder(root=dataset_path)

# View first image and its label
img_path, label = dataset.samples[2]

# Load actual image
image = Image.open(img_path).convert("RGB")

# Show it (optional)
image.show()

# Print label
print("Class:", dataset.classes[label])


Class: Angelina Jolie


In [40]:
print(dataset.class_to_idx)


{'Angelina Jolie': 0, 'Brad Pitt': 1, 'Denzel Washington': 2, 'Hugh Jackman': 3, 'Jennifer Lawrence': 4, 'Johnny Depp': 5, 'Kate Winslet': 6, 'Leonardo DiCaprio': 7, 'Megan Fox': 8, 'Natalie Portman': 9, 'Nicole Kidman': 10, 'Robert Downey Jr': 11, 'Sandra Bullock': 12, 'Scarlett Johansson': 13, 'Tom Cruise': 14, 'Tom Hanks': 15, 'Will Smith': 16}


In [41]:
for images, labels in loader:
    print(images.shape)  # torch.Size([32, 3, 224, 224])
    print(labels.shape)  # torch.Size([32])
    break


torch.Size([32, 3, 224, 224])
torch.Size([32])


In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import requests
import torch

# Choose an image embedding model (e.g., CLIP)
model_name = "openai/clip-vit-base-patch32"
# Load the processor (for image preprocessing) and the model
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# If you have a GPU, move the model to GPU for faster inference
if torch.cuda.is_available():
    model = model.cuda()

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

c:\Users\pvvis\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pvvis\.cache\huggingface\hub\models--openai--clip-vit-base-patch32. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [49]:
import os
import torch
import numpy as np
from PIL import Image
from torchvision import datasets
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm # For showing a progress bar

# --- Configuration ---
DATASET_PATH = r"D:\Facial-Media-Recognition-System\Dataset"
HF_MODEL_NAME = "openai/clip-vit-base-patch32" # Your chosen image embedding model

# --- 1. Load Hugging Face Model and Processor (Load once at start) ---
processor = None
model = None

try:
    print(f"Loading Hugging Face model: {HF_MODEL_NAME}...")
    processor = AutoProcessor.from_pretrained(HF_MODEL_NAME)
    model = AutoModel.from_pretrained(HF_MODEL_NAME)

    if torch.cuda.is_available():
        model = model.cuda()
        print("Model moved to GPU for faster processing.")
    else:
        print("Running model on CPU.")

    print(f"Successfully loaded {HF_MODEL_NAME}\n")

except Exception as e:
    print(f"CRITICAL ERROR: Failed to load model '{HF_MODEL_NAME}': {e}")
    print("Please ensure 'huggingface-cli login' is done if needed, and libraries are installed.")
    exit() # Exit if we can't load the model

# --- 2. Define the Embedding Generation Function ---
def generate_image_embedding(pil_image: Image.Image, processor, model) -> np.ndarray:
    """
    Generates an embedding for a given PIL Image using a Hugging Face processor and model.

    Args:
        pil_image (Image.Image): The PIL Image object to embed.
        processor: The loaded Hugging Face AutoProcessor object.
        model: The loaded Hugging Face AutoModel object.

    Returns:
        np.ndarray: The image embedding as a 1D NumPy array.
    """
    # Ensure the image is in RGB format
    rgb_pil_image = pil_image.convert("RGB")

    # Process the image with the model's AutoProcessor
    inputs = processor(images=rgb_pil_image, return_tensors="pt")

    # Move inputs to the same device as the model
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    # Generate embeddings with the model (disable gradient computation for inference)
    with torch.no_grad():
        image_features = model.get_image_features(**inputs) # For CLIP models

    # Convert to 1D NumPy array and return
    embedding = image_features.cpu().numpy().flatten()
    return embedding

# --- 3. Iterate Through the Entire Dataset and Generate Embeddings ---
if __name__ == "__main__":
    print(f"Preparing to process dataset from: {DATASET_PATH}")

    try:
        # Load the dataset using ImageFolder to get image paths and labels
        # No transform is needed here because PIL loads raw images and
        # the HF processor handles the transformations.
        dataset = datasets.ImageFolder(root=DATASET_PATH)

        if not dataset.samples:
            print(f"No image samples found in '{DATASET_PATH}'. Please check your dataset structure.")
            exit()

        all_embeddings = []
        all_labels = []
        all_image_paths = []

        print(f"Found {len(dataset.samples)} images in the dataset. Starting embedding generation...")

        # Use tqdm for a nice progress bar
        for idx, (img_path, label_idx) in enumerate(tqdm(dataset.samples, desc="Generating Embeddings")):
            try:
                # Load the image using PIL
                image = Image.open(img_path)

                # Generate the embedding
                embedding = generate_image_embedding(image, processor, model)

                if embedding is not None:
                    all_embeddings.append(embedding)
                    all_labels.append(label_idx)
                    all_image_paths.append(img_path)
                else:
                    print(f"Skipping embedding for {img_path} due to error.")

            except Exception as e:
                print(f"Error processing image {img_path}: {e}")
                print("Skipping this image.")
                continue

        # --- 4. Process and Store the Results ---
        if all_embeddings:
            # Convert list of embeddings to a single NumPy array
            final_embeddings_array = np.array(all_embeddings)
            final_labels_array = np.array(all_labels)

            print("\n--- Embedding Generation Complete ---")
            print(f"Total images processed: {len(all_embeddings)}")
            print(f"Shape of all_embeddings_array: {final_embeddings_array.shape}") # (num_images, embedding_dim)
            print(f"Shape of all_labels_array: {final_labels_array.shape}") # (num_images,)

            # You can now save these embeddings and labels for later use
            # Example: Save to .npy files
            output_dir = "generated_embeddings"
            os.makedirs(output_dir, exist_ok=True)
            embeddings_output_path = os.path.join(output_dir, "image_embeddings.npy")
            labels_output_path = os.path.join(output_dir, "image_labels.npy")
            paths_output_path = os.path.join(output_dir, "image_paths.txt") # Save paths as well

            np.save(embeddings_output_path, final_embeddings_array)
            np.save(labels_output_path, final_labels_array)
            with open(paths_output_path, 'w') as f:
                for path in all_image_paths:
                    f.write(f"{path}\n")

            print(f"Embeddings saved to: {embeddings_output_path}")
            print(f"Labels saved to: {labels_output_path}")
            print(f"Image paths saved to: {paths_output_path}")
            print(f"Dataset class names: {dataset.classes}")

            # Example: How to load them back later
            # loaded_embeddings = np.load(embeddings_output_path)
            # loaded_labels = np.load(labels_output_path)
            # print(f"\nExample of loading back: {loaded_embeddings.shape}")

        else:
            print("No embeddings were generated. Check for errors during processing.")

    except Exception as e:
        print(f"An unexpected error occurred during dataset processing: {e}")
        print("Please ensure your DATASET_PATH is correct and accessible.")

Loading Hugging Face model: openai/clip-vit-base-patch32...
Model moved to GPU for faster processing.
Successfully loaded openai/clip-vit-base-patch32

Preparing to process dataset from: D:\Facial-Media-Recognition-System\Dataset
Found 1800 images in the dataset. Starting embedding generation...


Generating Embeddings: 100%|██████████| 1800/1800 [00:30<00:00, 59.54it/s]


--- Embedding Generation Complete ---
Total images processed: 1800
Shape of all_embeddings_array: (1800, 512)
Shape of all_labels_array: (1800,)
Embeddings saved to: generated_embeddings\image_embeddings.npy
Labels saved to: generated_embeddings\image_labels.npy
Image paths saved to: generated_embeddings\image_paths.txt
Dataset class names: ['Angelina Jolie', 'Brad Pitt', 'Denzel Washington', 'Hugh Jackman', 'Jennifer Lawrence', 'Johnny Depp', 'Kate Winslet', 'Leonardo DiCaprio', 'Megan Fox', 'Natalie Portman', 'Nicole Kidman', 'Robert Downey Jr', 'Sandra Bullock', 'Scarlett Johansson', 'Tom Cruise', 'Tom Hanks', 'Will Smith']


In [48]:
image_embedding.shape

(512,)

In [50]:
import numpy as np

embeddings = np.load("generated_embeddings/image_embeddings.npy")
labels = np.load("generated_embeddings/image_labels.npy")
with open("generated_embeddings/image_paths.txt") as f:
    paths = f.read().splitlines()


In [58]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [60]:

# Load and embed the query image
query_path = r"D:\Facial-Media-Recognition-System\test\Test.jpg"
query_image = Image.open(query_path).convert("RGB")

inputs = processor(images=query_image, return_tensors="pt").to(device)

with torch.no_grad():
    query_embedding = model.get_image_features(**inputs).cpu().numpy()[0]  # shape: (512,)
